In [59]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

In [2]:
dados = pd.read_parquet('../../data/dados_modelo/base_modelo.parquet')
dados.head()

,id_municipio,id_escola,chave_rede,alfabetizado,proficiencia,peso_aluno,taxa_alfabetizacao_mun,media_portugues,sigla_uf,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE
0,5101837,None,None,None,NaN,NaN,NaN,NaN,MT,NaN,8.46,NaN
1,1100205,60000002,2,Sim,746.91,1.00,57.45,747.2139,RO,58.65,4.36,4.959
2,1100205,60000386,2,Não,727.29,1.00,57.45,747.2139,RO,58.65,4.36,4.959
3,1100205,60000386,2,Sim,764.40,1.00,57.45,747.2139,RO,58.65,4.36,4.959
4,1100205,60000002,2,Não,739.06,1.05,57.45,747.2139,RO,58.65,4.36,4.959


In [3]:
dados.isna().sum()

id_municipio                 0
id_escola                 5140
chave_rede                  21
alfabetizado              5140
proficiencia              5140
peso_aluno                5140
taxa_alfabetizacao_mun      21
media_portugues             21
sigla_uf                     0
taxa_alfabetizacao_uf       21
indice_analf                 0
MEDIA_INSE                1467
dtype: int64

In [4]:
df = dados.copy()

In [5]:
df = df[df['alfabetizado'].notna()]
df.isna().sum()


id_municipio                 0
id_escola                    0
chave_rede                   0
alfabetizado                 0
proficiencia                 0
peso_aluno                   0
taxa_alfabetizacao_mun       0
media_portugues              0
sigla_uf                     0
taxa_alfabetizacao_uf        0
indice_analf                 0
MEDIA_INSE                1429
dtype: int64

In [6]:
dir_uf = pd.read_parquet('../../data/dir_uf.parquet')
dir_uf

,id_uf,sigla_uf
0,51,MT
1,11,RO
2,12,AC
3,13,AM
4,14,RR
5,15,PA
6,16,AP
7,17,TO
8,21,MA
9,22,PI


In [7]:
df

,id_municipio,id_escola,chave_rede,alfabetizado,proficiencia,peso_aluno,taxa_alfabetizacao_mun,media_portugues,sigla_uf,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE
1,1100205,60000002,2,Sim,746.91,1.00,57.45,747.2139,RO,58.65,4.36,4.959
2,1100205,60000386,2,Não,727.29,1.00,57.45,747.2139,RO,58.65,4.36,4.959
3,1100205,60000386,2,Sim,764.40,1.00,57.45,747.2139,RO,58.65,4.36,4.959
4,1100205,60000002,2,Não,739.06,1.05,57.45,747.2139,RO,58.65,4.36,4.959
5,1100205,60000012,2,Não,736.43,1.21,57.45,747.2139,RO,58.65,4.36,4.959
...,...,...,...,...,...,...,...,...,...,...,...,...
1821405,5220405,60041783,3,Sim,758.16,1.11,72.78,757.9946,GO,66.72,8.28,5.413
1821406,5220405,60041698,3,Sim,756.65,1.14,72.78,757.9946,GO,66.72,8.28,5.413
1821407,5220405,60041698,3,Sim,772.79,1.15,72.78,757.9946,GO,66.72,8.28,5.413
1821408,5220405,60041698,3,Sim,771.91,1.00,72.78,757.9946,GO,66.72,8.28,5.413


Apenas a coluna "MEDIA_INSE" contém dados vazios. Acertaremos isso no pipeline, no momento de imputação dos valores restantes.

In [8]:
# Mapeamento do Alvo (Target Encoding Binário)
if 'alfabetizado' in df.columns:
    target_map = {'Sim': 1, 'Não': 0}
    df['alfabetizado'] = df['alfabetizado'].map(target_map)

In [9]:
uf_map = dir_uf.set_index('sigla_uf')['id_uf'].to_dict()
    
# Submeter o mapeamento e criar (ou substituir) a coluna
df['id_uf'] = df['sigla_uf'].map(uf_map)
df['id_uf'] = pd.to_numeric(df['id_uf'], errors='coerce')

# (Opcional) Se quiser remover a sigla antiga e manter só o ID:
df = df.drop(columns=['sigla_uf'])

df

,id_municipio,id_escola,chave_rede,alfabetizado,proficiencia,peso_aluno,taxa_alfabetizacao_mun,media_portugues,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE,id_uf
1,1100205,60000002,2,1,746.91,1.00,57.45,747.2139,58.65,4.36,4.959,11
2,1100205,60000386,2,0,727.29,1.00,57.45,747.2139,58.65,4.36,4.959,11
3,1100205,60000386,2,1,764.40,1.00,57.45,747.2139,58.65,4.36,4.959,11
4,1100205,60000002,2,0,739.06,1.05,57.45,747.2139,58.65,4.36,4.959,11
5,1100205,60000012,2,0,736.43,1.21,57.45,747.2139,58.65,4.36,4.959,11
...,...,...,...,...,...,...,...,...,...,...,...,...
1821405,5220405,60041783,3,1,758.16,1.11,72.78,757.9946,66.72,8.28,5.413,52
1821406,5220405,60041698,3,1,756.65,1.14,72.78,757.9946,66.72,8.28,5.413,52
1821407,5220405,60041698,3,1,772.79,1.15,72.78,757.9946,66.72,8.28,5.413,52
1821408,5220405,60041698,3,1,771.91,1.00,72.78,757.9946,66.72,8.28,5.413,52


In [10]:
# Remoção de Colunas Indesejadas e Vazamentos de Dados (Data Leakage)
cols_para_remover = [
        'id_escola',        # Vimos que este dado muda de um ano para outro
        'proficiencia',     # VAZAMENTO ABSOLUTO: Nota do próprio aluno no exame
        'peso_aluno',       # Peso estatístico amostral, sem valor preditivo individual
        'media_portugues'   # Nota já compõe o índice de proficiência, traz vazamento.
    ]

In [11]:
# Garante remoção apenas das colunas existentes na base
existing_cols_to_drop = [col for col in cols_para_remover if col in df.columns]
df = df.drop(columns=existing_cols_to_drop)

In [12]:
# Imputa os valores ausentes com a mediana por UF
df['MEDIA_INSE'] = df['MEDIA_INSE'].fillna(
    df.groupby('id_uf')['MEDIA_INSE'].transform('median')
)

In [13]:
df['MEDIA_INSE'] = pd.to_numeric(df['MEDIA_INSE'], errors='coerce')
df['indice_analf'] = pd.to_numeric(df['indice_analf'], errors='coerce')

inse = df['MEDIA_INSE']
analf = df['indice_analf'].fillna(0)
    
# Criar a nova variável não-linear
df['razao_inse_analf'] = np.divide(inse, analf + 1e-5)

df

,id_municipio,chave_rede,alfabetizado,taxa_alfabetizacao_mun,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE,id_uf,razao_inse_analf
1,1100205,2,1,57.45,58.65,4.36,4.959,11,1.137383
2,1100205,2,0,57.45,58.65,4.36,4.959,11,1.137383
3,1100205,2,1,57.45,58.65,4.36,4.959,11,1.137383
4,1100205,2,0,57.45,58.65,4.36,4.959,11,1.137383
5,1100205,2,0,57.45,58.65,4.36,4.959,11,1.137383
...,...,...,...,...,...,...,...,...,...
1821405,5220405,3,1,72.78,66.72,8.28,5.413,52,0.653743
1821406,5220405,3,1,72.78,66.72,8.28,5.413,52,0.653743
1821407,5220405,3,1,72.78,66.72,8.28,5.413,52,0.653743
1821408,5220405,3,1,72.78,66.72,8.28,5.413,52,0.653743


In [14]:
df.isna().sum()

id_municipio              0
chave_rede                0
alfabetizado              0
taxa_alfabetizacao_mun    0
taxa_alfabetizacao_uf     0
indice_analf              0
MEDIA_INSE                0
id_uf                     0
razao_inse_analf          0
dtype: int64

Resolvido o problema da coluna MEDIA_INSE

In [15]:
# Seleção das colunas após limpeza
colunas_numericas = ['taxa_alfabetizacao_mun','taxa_alfabetizacao_uf', 'indice_analf', 'MEDIA_INSE', 'razao_inse_analf']
cat_baixa_card = ['chave_rede']
cat_alta_card = ['id_municipio','id_uf']
coluna_alvo = 'alfabetizado'

In [16]:
from scipy.stats import spearmanr, pointbiserialr, chi2_contingency
from sklearn.feature_selection import mutual_info_classif

In [17]:
# Pipeline Numérico: Imputação por Mediana + Escalonamento Robusto contra Outliers
num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])

# Pipeline Categórico para Baixa Cardinalidade: Imputação por Moda + One-Hot Encoding
cat_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# Pipeline Categórico para Alta Cardinalidade: Imputação por Moda + Target Encoding
cat_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_enc', TargetEncoder(smooth="auto", cv=5))
])

In [18]:
# Montagem do ColumnTransformer Unificado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, colunas_numericas),
        ('cat_low', cat_low_transformer, cat_baixa_card),
        ('cat_high', cat_high_transformer, cat_alta_card)
    ],
    remainder='drop'
)

In [19]:
X = df.drop(columns=[coluna_alvo])
y = df[coluna_alvo]

In [20]:
# Divisão Treino/Teste Estratificada (ANTES de qualquer imputação ou transformação)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [21]:
# --- A. Teste de Correlação (Spearman e Pearson/Point-Biserial) ---
correlations = []
for col in colunas_numericas:
    spearman_corr, spearman_p = spearmanr(X_train[col], y_train)
    pb_corr, pb_p = pointbiserialr(X_train[col], y_train)
    
    correlations.append({
        'Feature': col,
        'Pearson/PointBiserial': pb_corr,
        'p-valor (Pearson)': pb_p,
        'Spearman': spearman_corr,
        'p-valor (Spearman)': spearman_p
    })

df_corr = pd.DataFrame(correlations)
print("=== CORRELAÇÃO DE VARIÁVEIS NUMÉRICAS ===")
print(df_corr.sort_values(by='Spearman', key=abs, ascending=False))

# --- B. Teste Qui-Quadrado para Categóricas ---
chi2_results = []
for col in cat_baixa_card:
    contingency_table = pd.crosstab(X_train[col].astype(str), y_train)
    chi2, p, dof, _ = chi2_contingency(contingency_table)
    chi2_results.append({
        'Feature': col,
        'Chi2 Statistic': chi2,
        'p-valor': p
    })

df_chi2 = pd.DataFrame(chi2_results)
print("\n=== TESTE QUI-QUADRADO (CATEGÓRICAS) ===")
print(df_chi2)

# --- C. Mutual Information (Numéricas + Categóricas) ---
# Prepara dados para o Mutual Info (exige tratar NaNs)
X_train_mi = X_train.copy()
mi_scores = mutual_info_classif(X_train_mi, y_train, random_state=42)

df_mi = pd.DataFrame({
    'Feature': X_train_mi.columns,
    'Mutual Information': mi_scores
}).sort_values(by='Mutual Information', ascending=False)

print("\n=== MUTUAL INFORMATION ===")
print(df_mi)

=== CORRELAÇÃO DE VARIÁVEIS NUMÉRICAS ===
                  Feature  Pearson/PointBiserial  p-valor (Pearson)  Spearman  \
0  taxa_alfabetizacao_mun               0.245809           0.000000  0.239046   
1   taxa_alfabetizacao_uf               0.200612           0.000000  0.189183   
3              MEDIA_INSE               0.063451           0.000000  0.062476   
2            indice_analf              -0.002618           0.001604  0.003601   
4        razao_inse_analf               0.001653           0.046288  0.003299   

   p-valor (Spearman)  
0            0.000000  
1            0.000000  
3            0.000000  
2            0.000014  
4            0.000070  

=== TESTE QUI-QUADRADO (CATEGÓRICAS) ===
      Feature  Chi2 Statistic        p-valor
0  chave_rede      803.696544  8.479459e-177

=== MUTUAL INFORMATION ===
                  Feature  Mutual Information
1              chave_rede            0.171598
7        razao_inse_analf            0.054866
0            id_municipio    

In [22]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# 1. PASSO 1: Definir o seu full_pipeline (exatamente como você já fez)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=100,       # Será sobrescrito pelo RandomizedSearchCV
        learning_rate=0.1,      # Será sobrescrito pelo RandomizedSearchCV
        random_state=42,
        eval_metric='logloss'
    ))
])

# 2. PASSO 2: Definir o espaço de busca (usando o prefixo 'classifier__')
param_distributions = {
    'classifier__n_estimators': randint(100, 500),
    'classifier__max_depth': randint(3, 8),
    'classifier__learning_rate': uniform(0.01, 0.2),
    'classifier__subsample': uniform(0.6, 0.4),
    'classifier__colsample_bytree': uniform(0.6, 0.4),
    'classifier__min_child_weight': randint(1, 6)
}

# 3. PASSO 3: Passar o seu full_pipeline para o RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=full_pipeline,       # <-- Seu pipeline entra aqui
    param_distributions=param_distributions,
    n_iter=25,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)

# 4. PASSO 4: Treinar e buscar os melhores hiperparâmetros
search.fit(X_train, y_train)

KeyboardInterrupt: 

In [54]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

In [55]:
# O fit calcula a mediana/estatísticas APENAS em X_train e treina o modelo
full_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat_low', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different t

In [56]:
# O predict aplica o transformador em X_test usando as estatísticas do X_train
accuracy = full_pipeline.score(X_test, y_test)
print(f"Acurácia no Conjunto de Teste: {accuracy:.4f}")

Acurácia no Conjunto de Teste: 0.6403


In [57]:
# Verificar a proporção da sua variável alvo no treino e teste:
print(y_test.value_counts(normalize=True))

alfabetizado
1    0.598276
0    0.401724
Name: proportion, dtype: float64


In [ ]:
tree_model = full_pipeline.named_steps['classifier']
plot_tree(tree_model)

KeyboardInterrupt: 